In [13]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import librosa
import librosa.display
import matplotlib.pyplot as plt
import random
import torch
from pathlib import Path


In [14]:
#----------------------------- DON'T CHANGE THIS --------------------------
DATA_SEED = 67
TRAINING_SEED = 1234
SR = 22050
DURATION = 5.0
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
TOP_DB=20
TARGET_SNR_DB = 10

random.seed(DATA_SEED)
np.random.seed(DATA_SEED)
torch.manual_seed(DATA_SEED)
torch.cuda.manual_seed(DATA_SEED)

In [15]:
DATA_ROOT = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"
GENRES = ["blues", "classical", "country", "disco", "hiphop",
          "jazz", "metal", "pop", "reggae", "rock"]
STEMS = {
    'drums.wav': 'drums',
    'vocals.wav': 'vocals',
    'bass.wav': 'bass',
    'other.wav': 'other'
}
STEM_KEYS = ['drums', 'vocals', 'bass', 'other']
GENRE_TO_TEST = 'rock'

In [16]:
def build_full_dataset(root_dir):
    full_dataset = {g: {k: [] for k in STEM_KEYS} for g in GENRES}

    for genre in GENRES:
        genre_path = os.path.join(root_dir, genre)
        songs = sorted(os.listdir(genre_path))

        for song in songs:
            song_path = os.path.join(genre_path, song)
            stem_files = []

            for stem_file in STEMS:
                fpath = os.path.join(song_path, stem_file)
                if not os.path.exists(fpath):
                    stem_files = []
                    break
                stem_files.append(fpath)

            if len(stem_files) == 4:
                for stem_file in STEMS:
                    full_dataset[genre][STEMS[stem_file]].append(os.path.join(song_path, stem_file))

    return full_dataset

In [ ]:
def get_duration(file_path):
    try:
        y, sr = librosa.load(file_path, sr=None, mono=True)
        return librosa.get_duration(y=y, sr=sr)
    except:
        return None

def get_sample_rate(file_path):
    try:
        return librosa.get_samplerate(file_path)
    except:
        return None

def get_peak_db(file_path):
    try:
        y, sr = librosa.load(file_path, sr=None, mono=True)
        peak = np.max(np.abs(y))
        if peak == 0:
            return -np.inf
        return 20 * np.log10(peak)
    except:
        return None

def get_spectral_centroid(file_path):
    try:
        y, sr = librosa.load(file_path, sr=None, mono=True)
        centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
        return np.mean(centroid)
    except:
        return None

def has_silence_first_half_second(file_path):
    try:
        y, sr = librosa.load(file_path, sr=None, mono=True)
        first_half_sec = y[:int(0.5 * sr)]
        if len(first_half_sec) == 0:
            return False
        return np.max(np.abs(first_half_sec)) < 1e-4
    except:
        return False


def answer_milestone_questions(dataset):

    jazz_durations = []
    for stem in STEM_KEYS:
        for f in dataset['jazz'][stem]:
            d = get_duration(f)
            if d is not None:
                jazz_durations.append(d)
    print("Q1 Mean Jazz Duration (seconds):", np.mean(jazz_durations))

    sample_rates = set()
    for genre in dataset:
        for stem in STEM_KEYS:
            for f in dataset[genre][stem]:
                sr = get_sample_rate(f)
                if sr:
                    sample_rates.add(sr)
    print("Q2 Unique Sample Rates:", sorted(sample_rates))

    corrupted_count = 0
    for genre in dataset:
        for stem in STEM_KEYS:
            for f in dataset[genre][stem]:
                if os.path.getsize(f) == 0:
                    corrupted_count += 1
    print("Q3 Corrupted files:", corrupted_count)

    vocal_peaks = []
    for genre in dataset:
        for f in dataset[genre]['vocals']:
            db = get_peak_db(f)
            if db is not None:
                vocal_peaks.append(db)
    print("Q4 Avg Peak Amplitude (dB) vocals:", np.mean(vocal_peaks))

    blues_centroids = []
    for stem in STEM_KEYS:
        for f in dataset['blues'][stem]:
            c = get_spectral_centroid(f)
            if c is not None:
                blues_centroids.append(c)
    print("Q5 Mean Spectral Centroid (blues):", np.mean(blues_centroids))

    genre_centroid_means = {}
    for genre in dataset:
        centroids = []
        for stem in STEM_KEYS:
            for f in dataset[genre][stem]:
                c = get_spectral_centroid(f)
                if c is not None:
                    centroids.append(c)
        genre_centroid_means[genre] = np.mean(centroids)
    print("Q6 Genre with highest mean spectral centroid:", max(genre_centroid_means, key=genre_centroid_means.get))

    silence_count = 0
    for genre in dataset:
        for stem in STEM_KEYS:
            for f in dataset[genre][stem]:
                if has_silence_first_half_second(f):
                    silence_count += 1
    print("Q7 Files with silence in first 0.5s:", silence_count)

In [ ]:
full_dataset = build_full_dataset(DATA_ROOT)
answer_milestone_questions(full_dataset)

# page 2

In [17]:
import os
import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, confusion_matrix, classification_report

In [18]:
ROOT = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
STEMS_PATH = os.path.join(ROOT, 'genres_stems')
GENRES = ["blues", "classical", "country", "disco", "hiphop", "jazz", "metal", "pop", "reggae", "rock"]

In [19]:
def extract_features(song_path):
    y, sr = librosa.load(os.path.join(song_path, 'other.wav'), sr=22050, duration=10)
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    spec_cent = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
    zcr = np.mean(librosa.feature.zero_crossing_rate(y))
    rolloff = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))
    return [float(tempo), spec_cent, zcr, rolloff]

data = []
for g in GENRES:
    gp = os.path.join(STEMS_PATH, g)
    songs = [s for s in os.listdir(gp) if os.path.isdir(os.path.join(gp, s))]
    for s in songs[:50]:
        data.append({'path': os.path.join(gp, s), 'genre': g})

df = pd.DataFrame(data)
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['genre'], random_state=42)

X_train = np.array([extract_features(p) for p in train_df['path']])
y_train = train_df['genre']
X_val = np.array([extract_features(p) for p in val_df['path']])
y_val = val_df['genre']

clf = DecisionTreeClassifier(max_depth=5, random_state=42)
clf.fit(X_train, y_train)

KeyboardInterrupt: 

In [ ]:
y_pred = clf.predict(X_val)

macro_f1 = f1_score(y_val, y_pred, average='macro')
cm = confusion_matrix(y_val, y_pred, labels=GENRES)
cr = classification_report(y_val, y_pred)
accuracy = np.mean(y_pred == y_val)

print(f"Validation Macro F1 Score: {macro_f1:.4f}\n")
print("Detailed Classification Report:")
print(cr)
print(f"Model Accuracy: {accuracy:.4f}")

In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=GENRES,
            yticklabels=GENRES,
            cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

tp_fp_fn_tn = {}

for i, genre in enumerate(GENRES):
    TP = cm[i, i]
    FP = cm[:, i].sum() - TP
    FN = cm[i, :].sum() - TP
    TN = cm.sum() - (TP + FP + FN)
    tp_fp_fn_tn[genre] = {"TP": TP, "FP": FP, "FN": FN, "TN": TN}

print("TP, FP, FN, TN per Genre:")
for g in tp_fp_fn_tn:
    print(g, tp_fp_fn_tn[g])